# Module 04 — Structured Output

So far every answer came back as a **string** (`.text`). Strings are fine for humans but painful for code — to *use* an answer you'd have to parse it, and that parsing breaks the moment the model rephrases.

**Structured output** fixes this: you declare a **schema** (the exact fields and types you want), and LangChain forces the model to return a real object that matches. No parsing, guaranteed shape.

Run top to bottom.

## 0. Setup

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join("..", ".env"))
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

from langchain.chat_models import init_chat_model
model = init_chat_model("google_genai:gemini-2.5-flash", temperature=0)

print("Key loaded:", bool(os.getenv("GOOGLE_API_KEY")))

Key loaded: True


## 1. The problem: a string you have to parse

Ask for structured info and you still get prose. To use it in code you'd hunt through the text — fragile and ugly. Run this and notice: useful info, but trapped in a sentence.

In [2]:
text = "The new iPhone 15 costs $799 and weighs 171 grams."

print(model.invoke("Extract the product name, price, and weight from: " + text).text)
# To USE these values you'd now have to string-parse this. No thanks.

*   **Product Name:** iPhone 15
*   **Price:** $799
*   **Weight:** 171 grams


## 2. Define a schema with Pydantic

A **schema** is a class that declares the fields you want and their types. We use **Pydantic** (`BaseModel`). The `Field(description=...)` text isn't just docs — the model reads it to know what each field means, so write it clearly.

This `Product` schema says: I want a string `name`, a float `price_usd`, and an int `weight_grams`.

In [4]:
from pydantic import BaseModel, Field

class Product(BaseModel):
    name: str = Field(description="The product's name")
    price_usd: float = Field(description="Price in US dollars, just the number")
    weight_grams: int = Field(description="Weight in grams, just the number")

## 3. `with_structured_output` — get a real object back

`model.with_structured_output(Product)` returns a new model that, when invoked, gives you a **`Product` instance** instead of a message. Now you access fields directly: `result.price_usd`. No parsing.

Notice we call `.invoke` as usual — only the *return type* changed.

In [5]:
extractor = model.with_structured_output(Product)

result = extractor.invoke("The new iPhone 15 costs $799 and weighs 171 grams.")

print("Type:", type(result).__name__)
print("Name:", result.name)
print("Price:", result.price_usd, "(", type(result.price_usd).__name__, ")")
print("Weight:", result.weight_grams)
print("Price with tax:", round(result.price_usd * 1.1, 2))  # it's a real number!

Type: Product
Name: iPhone 15
Price: 799.0 ( float )
Weight: 171
Price with tax: 878.9


## 4. Constrain a field to fixed choices

Often you want a field to be one of a *fixed set* of values — a classic classification. Use `Literal` to lock it down, so the model can't return "kinda positive". 

Here we classify a review's sentiment and pull out the topics as a **list of strings**.

In [6]:
from typing import Literal

class Review(BaseModel):
    sentiment: Literal["positive", "negative", "mixed"] = Field(description="Overall sentiment")
    topics: list[str] = Field(description="What aspects the review mentions, e.g. battery, price")
    summary: str = Field(description="One short sentence summarizing the review")

review_model = model.with_structured_output(Review)
out = review_model.invoke(
    "I love the battery life but the camera is disappointing and the price is too high."
)
print(out)
print("Sentiment:", out.sentiment)
print("Topics:", out.topics)

sentiment='mixed' topics=['battery life', 'camera', 'price'] summary='The battery life is great, but the camera is disappointing and the price is too high.'
Sentiment: mixed
Topics: ['battery life', 'camera', 'price']


## 5. Nested schemas & lists of objects

Schemas compose. A field can be a **list of another schema**, letting you extract many structured items at once. Here one sentence mentions several people; we get back a clean list of `Person` objects.

In [7]:
class Person(BaseModel):
    name: str = Field(description="Person's name")
    role: str = Field(description="Their job or role")

class Team(BaseModel):
    people: list[Person] = Field(description="Everyone mentioned")

team_model = model.with_structured_output(Team)
team = team_model.invoke(
    "Ada is the lead engineer, Grace handles design, and Alan runs the project."
)
for p in team.people:
    print(f"- {p.name}: {p.role}")

- Ada: lead engineer
- Grace: design
- Alan: project manager


## Recap

- `.text` is a string; using it in code means fragile parsing. **Structured output** returns a typed object instead.
- Declare a **schema** (Pydantic `BaseModel`) with typed fields and clear `Field(description=...)` — the model reads those descriptions.
- `model.with_structured_output(Schema)` → `.invoke(...)` returns a **schema instance**; access fields directly.
- Use **`Literal`** to restrict a field to fixed choices; use **`list[...]`** for multiple values.
- Schemas **nest**: a field can be a list of other schemas to extract many objects at once.

## 🛠️ Try it yourself

1. Add a `currency: str` field to `Product` and re-extract. Watch the model fill it.
2. Make a schema for a recipe (`title`, `ingredients: list[str]`, `minutes: int`) and extract one from a paragraph you write.
3. Change a `Field` description to be vague vs. precise and see if extraction quality changes.
4. Add `Literal["easy", "medium", "hard"]` difficulty to the recipe schema.
5. **Stretch:** make a schema with an *optional* field (`from typing import Optional`) and feed text that's missing that info — see what comes back.

When you're done, say **"next"** and we'll build **Module 05 — Tools**.